# 13 · 1-D Dynamic Programming

> La DP serve quando la stessa sottodomanda si ripete: la risolvi una volta e salvi il risultato. Nella DP 1D lo **stato** è un solo indice (`dp[i]` = risposta per i primi `i` elementi, o per l'importo `i`).
> Procedura e idee: [Pattern algoritmici › Programmazione dinamica](../Fundamentals/Pattern%20algoritmici.md#programmazione-dinamica) · [DP 1D](../Fundamentals/Pattern%20algoritmici.md#dp-1d)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Climbing Stairs | `dp[i]` dipende dai due precedenti | due variabili `a, b = b, a + b` |
| Min Cost Climbing Stairs | come sopra, con un costo e `min` | `min(...)`, attenzione a dove "finisce" la scala |
| House Robber | prendi la casa (+ due indietro) o saltala | `max(prendi, salta)` |
| House Robber II | case in cerchio → due problemi lineari | `nums[1:]`, `nums[:-1]`, caso `len == 1` |
| Longest Palindromic Substring | espandi attorno a ogni centro | `while l >= 0 and r < n and s[l] == s[r]` |
| Palindromic Substrings | stessa espansione, conta invece di tenere il massimo | `2n − 1` centri |
| Decode Ways | `dp[i]` da una o due cifre | `s[i - 1] != "0"`, `10 <= int(s[i-2:i]) <= 26` |
| Coin Change | minimo numero di monete per ogni importo | `[inf] * (amount + 1)`, `-1` se resta `inf` |
| Maximum Product Subarray | tieni massimo **e** minimo che finiscono in `i` | tupla di tre candidati |
| Word Break | `dp[i]` = il prefisso di lunghezza `i` si può comporre | `set(wordDict)`, `s[j:i] in words` |
| Longest Increasing Subsequence | `O(n²)` con `dp[i]`, o `O(n log n)` con `bisect` | `bisect_left(tails, x)` |
| Partition Equal Subset Sum | zaino booleano: somme raggiungibili | set di somme, `total % 2` |

## Sintassi al volo

| Serve | Sintassi |
|---|---|
| memoization automatica | `from functools import cache` · `@cache` sopra la funzione ricorsiva |
| tabella 1D | `dp = [0] * (n + 1)` · `dp = [float("inf")] * (n + 1)` |
| solo gli ultimi due valori | `prev2, prev1 = prev1, max(prev1, prev2 + x)` |
| sottostringa a due cifre | `int(s[i - 2 : i])` |
| svuotare la cache tra test | `func.cache_clear()` |
| ricorsione profonda | `sys.setrecursionlimit(10**4)` |

## Dalla ricorsione alla DP

1. Scrivi la **ricorrenza**: la risposta per `i` in funzione di risposte più piccole.
2. **Top-down:** la scrivi ricorsiva e aggiungi `@cache`. È la via più veloce da scrivere.
3. **Bottom-up:** riempi un array dai casi base in su. Nessun limite di ricorsione.
4. **Ottimizza lo spazio:** se `dp[i]` usa solo `dp[i-1]` e `dp[i-2]`, bastano due variabili.

In [1]:
from functools import cache

# In quanti modi posso comporre n unità usando confezioni da 1 o da 2?
@cache
def ways(n: int) -> int:
    if n <= 1:
        return 1
    return ways(n - 1) + ways(n - 2)

def ways_table(n: int) -> int:
    dp = [0] * (n + 1)
    dp[0] = dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]

def ways_two_vars(n: int) -> int:
    a, b = 1, 1  # dp[i-2], dp[i-1]
    for _ in range(n - 1):
        a, b = b, a + b  # il lato destro è valutato prima dell'assegnazione
    return b

print(ways(10), ways_table(10), ways_two_vars(10))  # -> 89 89 89

89 89 89


⚠️ Con `@cache` gli argomenti devono essere hashable: passa indici o tuple, non liste. Su LeetCode, dentro un metodo, definisci la funzione annidata così la cache è per singola chiamata.

## Prendi o salta

Molti problemi 1D hanno due scelte per l'elemento `i`: usarlo (e rispettare un vincolo, es. non usare il precedente) o saltarlo. La risposta è il `max`/`min` delle due.

In [2]:
# Promozioni giornaliere: non si possono attivare due giorni consecutivi. Massimo incasso?
daily_bonus = [200, 700, 900, 300, 100]

take_prev2, best_prev1 = 0, 0
for bonus in daily_bonus:
    take_prev2, best_prev1 = best_prev1, max(best_prev1, take_prev2 + bonus)
print(best_prev1)  # -> 1200 (200 + 900 + 100)

# Per una sequenza circolare: risolvi due volte, senza il primo e senza l'ultimo
print(daily_bonus[1:], daily_bonus[:-1])  # -> [700, 900, 300, 100] [200, 700, 900, 300]

1200
[700, 900, 300, 100] [200, 700, 900, 300]


## DP sull'importo (zaino illimitato)

`dp[x]` = risposta per l'importo `x`. Per ogni `x` provi ogni moneta `c ≤ x` e guardi `dp[x - c]`. Valore iniziale `inf` per "irraggiungibile".

In [3]:
coupons = [10, 30, 40]
amount = 60
dp = [0] + [float("inf")] * amount
for x in range(1, amount + 1):
    for c in coupons:
        if c <= x:
            dp[x] = min(dp[x], dp[x - c] + 1)
print(dp[amount], dp[5])  # -> 2 inf: 30 + 30 (la scelta greedy 40 + 10 + 10 ne userebbe 3)

2 inf


## Stringhe: prefissi e centri

**Prefissi:** `dp[i]` riguarda `s[:i]` (i primi `i` caratteri), quindi `dp` ha lunghezza `n + 1` e `dp[0]` è la stringa vuota. Il carattere "corrente" è `s[i - 1]`.

In [4]:
words = {"mug", "mugs", "et", "set"}
s = "mugset"
ok = [True] + [False] * len(s)  # ok[i]: s[:i] si può comporre
for i in range(1, len(s) + 1):
    for j in range(i):
        if ok[j] and s[j:i] in words:
            ok[i] = True
            break
print(ok)  # -> [True, False, False, True, True, False, True]

code = "226"
i = 3
print(code[i - 1], code[i - 2 : i], 10 <= int(code[i - 2 : i]) <= 26)  # -> 6 26 True

[True, False, False, True, True, False, True]
6 26 True


**Centri di palindromi:** ogni palindromo ha un centro, che può essere un carattere (lunghezza dispari) o lo spazio tra due caratteri (lunghezza pari). Espandi finché i bordi coincidono: `O(n²)` tempo, `O(1)` spazio.

In [5]:
def expand(s: str, l: int, r: int) -> tuple[int, int]:
    while l >= 0 and r < len(s) and s[l] == s[r]:
        l -= 1
        r += 1
    return l + 1, r  # s[l+1 : r] è il palindromo (l e r sono andati un passo oltre)

s = "xabbay"
print(expand(s, 2, 2), expand(s, 2, 3))  # -> (2, 3) (1, 5)
start, end = expand(s, 2, 3)
print(s[start:end])  # -> abba

(2, 3) (1, 5)
abba


## Massimo e minimo insieme

Quando un valore negativo può trasformare il minimo nel massimo (prodotti), porta avanti entrambi. Calcola i nuovi valori da quelli **vecchi** in una sola assegnazione.

In [6]:
changes = [2, 3, -2, 4, -1]  # fattori moltiplicativi
cur_max = cur_min = best = changes[0]
for x in changes[1:]:
    candidates = (x, x * cur_max, x * cur_min)
    cur_max, cur_min = max(candidates), min(candidates)
    best = max(best, cur_max)
print(best)  # -> 48 (tutto il prodotto)

48


## Sottosequenza crescente: `O(n log n)` con `bisect`

`tails[k]` = il più piccolo valore finale possibile di una sottosequenza crescente lunga `k + 1`. Per ogni `x`, sostituisci il primo `tails[i] >= x` (o aggiungi in fondo). La lunghezza di `tails` è la risposta (il contenuto **non** è una sottosequenza valida).

In [7]:
from bisect import bisect_left

prices = [10, 9, 2, 5, 3, 7, 101, 18]
tails: list[int] = []
for x in prices:
    i = bisect_left(tails, x)
    if i == len(tails):
        tails.append(x)
    else:
        tails[i] = x
print(len(tails), tails)  # -> 4 [2, 3, 7, 18]

4 [2, 3, 7, 18]


## Set di somme raggiungibili

Per "esiste un sottoinsieme con somma S?" basta un set: per ogni elemento, le nuove somme sono le vecchie più l'elemento. Itera su una **copia** (o costruisci un nuovo set), altrimenti usi lo stesso elemento più volte.

In [8]:
weights = [1, 5, 11, 5]
total = sum(weights)
print(total % 2)  # -> 0: con somma dispari la divisione in due parti uguali è impossibile

reachable = {0}
for w in weights:
    reachable |= {s + w for s in reachable}  # il set a destra è costruito dal vecchio
print(total // 2 in reachable)  # -> True

0
True


## Errori comuni

- `dp` lungo `n` invece di `n + 1` quando lo stato è "primi `i` elementi".
- `[0] * n` con `float("inf")` dimenticato: un minimo inizializzato a 0 resta 0.
- Restituire `inf` invece di `-1` (Coin Change) o non gestire `"0"` iniziale (Decode Ways).
- Ottimizzazione dello spazio con aggiornamenti in ordine sbagliato: usa l'assegnazione multipla.
- `@cache` su un metodo con `self`: funziona, ma la cache sopravvive tra chiamate. Meglio una funzione annidata.